# Feature Engineering — Fraud Detection

Joins silver transactions with account + customer dimensions to build a
transaction-level feature table for fraud classification.

**Reads from:** `silver_transactions`, `silver_accounts`, `silver_customers`

**Writes to:** `gold_ml_features`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp, when, log1p

spark = SparkSession.builder.getOrCreate()
print('Spark session ready')

In [ ]:
txn = spark.read.table('silver_transactions')
acct = spark.read.table('silver_accounts')
cust = spark.read.table('silver_customers')
print(f'Transactions: {txn.count():,} | Accounts: {acct.count():,} | Customers: {cust.count():,}')

required = {'transaction_id', 'account_id', 'customer_id', 'amount', 'is_flagged_fraud'}
missing = required - set(txn.columns)
if missing:
    raise ValueError(f'silver_transactions missing columns {sorted(missing)}. Regenerate data and rerun bronze/silver.')

In [ ]:
# Join transaction + account + customer attributes. EXCLUDE silver leakage
# columns (risk_score / risk_band are derived from the fraud flag).
ml_features = (
    txn.select(
        'transaction_id', 'account_id', 'customer_id', 'transaction_date',
        'transaction_type', 'merchant_category', 'channel', 'country',
        'amount', 'transaction_hour',
        col('is_night_transaction').cast('int').alias('is_night'),
        col('is_international').cast('int').alias('is_international'),
        col('is_high_value').cast('int').alias('is_high_value'),
        col('is_flagged_fraud').cast('int').alias('had_fraud'),
    )
    .join(
        acct.select('account_id', 'account_type', 'balance', 'credit_limit', 'credit_utilisation_pct'),
        'account_id', 'left')
    .join(
        cust.select('customer_id', 'age_group', 'segment', 'region', 'risk_tier'),
        'customer_id', 'left')
    .withColumn('log_amount', log1p(col('amount')))
    .na.fill(0)
    .na.fill('unknown', subset=['transaction_type', 'merchant_category', 'channel',
                                'country', 'account_type', 'age_group', 'segment',
                                'region', 'risk_tier'])
    .withColumn('feature_timestamp', current_timestamp())
)

total_rows = ml_features.count()
positive_rows = ml_features.filter(col('had_fraud') == 1).count()
fraud_rate = (positive_rows / total_rows * 100) if total_rows else 0.0

# Guardrail: fail fast if positives collapse (silent label loss).
if total_rows < 1000 or positive_rows < max(10, int(total_rows * 0.01)):
    raise ValueError(
        f'Label quality check failed: only {positive_rows}/{total_rows} fraud rows '
        f'({fraud_rate:.2f}%). Check is_flagged_fraud typing and source data.'
    )

ml_features.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_features')
print(f'Gold ML features written: {total_rows:,} rows | fraud rate {fraud_rate:.1f}%')

In [ ]:
spark.sql('OPTIMIZE gold_ml_features')
print('Feature table optimized')